# CAGE 4: GitHub -> CPU + CUDA -> 500k
Chọn **Runtime > Change runtime type > T4 GPU**. Chạy từng cell từ trên xuống.
Không upload source/archive, không tự tạo RUN_ROOT, không mang .venv từ Mac sang.
Notebook clone toàn bộ nhánh main-benchmark và khóa commit để các seeds dùng cùng source.
Một seed chạy tuần tự 6 cấu hình: NodeMLP/GCN/GraphSAGE x Full/mixed-q.
Mỗi cấu hình 500k simulator ticks, validation tại 100k/200k/300k/400k/500k.

**Không có exact resume**: mất runtime không tự tiếp tục từ checkpoint.
Notebook không xóa/ghi đè run cũ. Chi phí CPU/GPU chỉ so sánh trên cùng cấu hình thiết bị.


In [ ]:
from pathlib import Path
import json
import os
import signal
import subprocess
import sys
import tempfile
import time

SEED = 42  # Chọn 42, 43, 44, 45 hoặc 46.
RUN_TAG = "github_v1"  # Chỉ đổi khi chủ động tạo đợt MỚI; không phải resume.
REPO_URL = "https://github.com/Felix-Think/Benchmarking-Graph-Neural-Network-Encoders-for-PPO-based-Autonomous-Cyber-Defense.git"
BRANCH = "main-benchmark"
SOURCE_COMMIT = "e2a55dc8ea6e4bed86eca9934c6421dab8c444e5"
REPO = Path("/content/acd_benchmark_github_" + SOURCE_COMMIT[:8])
CONFIG = Path("configs/experiments/encoder_retention_500k_cuda.json")
RUN_ROOT = Path(
    f"/content/drive/MyDrive/acd_runs/encoder_retention_500k_cuda_seed{SEED}_{RUN_TAG}"
)
QUALIFIED = None

def run_visible(command, cwd=None):
    """Stream merged stdout/stderr; fail immediately on command errors."""
    with subprocess.Popen(
        list(map(str, command)), cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            code = process.wait()
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            process.wait()
            raise
    if code:
        raise subprocess.CalledProcessError(code, command)

print("Seed:", SEED, "\nSource:", SOURCE_COMMIT, "\nOutput:", RUN_ROOT)
run_visible(["nvidia-smi"])


## 1. Clone đầy đủ source
Không pull hoặc đổi source giữa các seeds. Nếu thư mục đã tồn tại, chỉ dùng lại
khi đúng commit và không có thay đổi source. Nếu GitHub yêu cầu quyền truy cập,
dừng và cấp quyền bằng phương thức an toàn; không ghi token vào notebook.


In [ ]:
if not REPO.exists():
    run_visible(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO])
    run_visible(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO)
else:
    if not (REPO / ".git").exists():
        raise RuntimeError(f"Thư mục không phải Git clone: {REPO}; không tự ghi đè.")

head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
).strip()
if head != SOURCE_COMMIT:
    raise RuntimeError(f"Sai commit: {head}; yêu cầu {SOURCE_COMMIT}.")
changes = subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO, text=True
)
if changes.strip():
    raise RuntimeError("Source đã thay đổi; không tự reset hoặc trộn phiên bản:\n" + changes)

required = [
    ".python-version", "README.md", "pyproject.toml", "uv.lock",
    "docs/ENCODER_RETENTION_PROTOCOL.md", str(CONFIG),
    "scripts/experiments/run_encoder_matrix.py",
    "scripts/experiments/smoke_cuda.py", "tests/test_cuda_pipeline.py",
]
missing = [name for name in required if not (REPO / name).is_file()]
missing += [name for name in ("src", "scripts", "CybORG", "configs", "tests")
            if not (REPO / name).is_dir()]
if missing:
    raise RuntimeError(f"Source thiếu: {missing}")
config = json.loads((REPO / CONFIG).read_text())
if SEED not in config["seeds"]:
    raise ValueError(f"Seed {SEED} không thuộc {config['seeds']}")
assert config["training"]["device"] == "cuda"
assert config["training"]["total_simulator_steps"] == 500000
os.chdir(REPO)
print("Source đầy đủ:", REPO)
print("Seeds:", config["seeds"])


## 2. Cài uv và môi trường Python 3.10
Dùng môi trường đã khóa trong uv.lock, không dùng Torch của kernel Colab.
Cell này tự cài uv; không cần cài sẵn.


In [ ]:
run_visible([sys.executable, "-m", "pip", "install", "-q", "uv"])
# Gọi uv bằng module để không phụ thuộc PATH của Colab.
UV = [sys.executable, "-m", "uv"]
run_visible(UV + ["sync", "--locked", "--python", "3.10", "--group", "dev"], cwd=REPO)
run_visible(UV + [
    "run", "--no-sync", "python", "-c",
    "import sys, torch; print(sys.version); print('Torch:', torch.__version__); "
    "print('CUDA:', torch.cuda.is_available()); "
    "assert torch.cuda.is_available(), 'Chưa có NVIDIA CUDA GPU'; "
    "print('GPU:', torch.cuda.get_device_name(0))",
], cwd=REPO)


## 3. Kiểm thử CUDA trước khi train
Phải pass trên GPU thật cho cả ba encoder. Không bỏ qua cell này.
Kiểm thử ngắn dùng thư mục tạm, không tạo run benchmark trên Drive.


In [ ]:
QUALIFIED = None
run_visible(UV + [
    "run", "--no-sync", "python", "-m", "scripts.experiments.smoke_cuda",
], cwd=REPO)
QUALIFIED = SOURCE_COMMIT
print("CUDA smoke tests PASSED.")


## 4. Kết nối Drive và xem kế hoạch
Chỉ mount Drive, không mkdir RUN_ROOT. Nếu output đã tồn tại, notebook dừng.
Giữ run cũ để kiểm tra; không đổi tên để vô tình train trùng kết quả.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

if RUN_ROOT.exists():
    raise FileExistsError(
        f"Run đã tồn tại: {RUN_ROOT}. Không có auto-resume; kiểm tra trước khi tạo run mới."
    )

command = UV + [
    "run", "--no-sync", "python", "-u", "-m",
    "scripts.experiments.run_encoder_matrix",
    "--config", str(CONFIG),
    "--run-root", str(RUN_ROOT), "--seeds", str(SEED),
]
run_visible(command, cwd=REPO)  # Chỉ in kế hoạch, chưa train.


## 5. Bắt đầu train, hiển thị log trực tiếp
Cell này chạy thật và chờ cả 6 cấu hình hoàn thành. Chỉ bấm một lần.
Log update in mỗi 50 updates; validation/rollout có thể mất thời gian giữa hai dòng log.
Khi bấm Stop, notebook gửi tín hiệu dừng cho queue để cleanup workers;
không xem đó là pause/resume. Mất runtime có thể mất phần chưa lưu.


In [ ]:
if QUALIFIED != SOURCE_COMMIT:
    raise RuntimeError("Chạy CUDA smoke test thành công trước.")
if RUN_ROOT.exists():
    raise FileExistsError(f"Không ghi đè run đã có: {RUN_ROOT}")

def read_status():
    path = RUN_ROOT / "status.json"
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        return {}

offsets = {}
def print_new_log(path):
    path = Path(path)
    if not path.is_file():
        return
    offset = offsets.get(path, 0)
    with path.open("rb") as stream:
        stream.seek(offset)
        data = stream.read(65536)
        offsets[path] = stream.tell()
    if data:
        print(data.decode("utf-8", errors="replace"), end="", flush=True)

def display_progress():
    print_new_log(queue_log)
    status = read_status()
    # Read both completed and current logs so the last lines aren't lost
    # when the queue moves to the next model.
    jobs = [*status.get("completed", [])]
    if status.get("current"):
        jobs.append(status["current"])
    for job in jobs:
        print_new_log(Path(job["directory"]) / "console.log")

with tempfile.NamedTemporaryFile(prefix="acd_queue_", suffix=".log", delete=False) as file:
    queue_log = Path(file.name)

print("Starting:", RUN_ROOT, flush=True)
with queue_log.open("wb") as output:
    process = subprocess.Popen(
        command + ["--foreground"], cwd=REPO,
        stdout=output, stderr=subprocess.STDOUT, start_new_session=True,
    )
    try:
        while process.poll() is None:
            display_progress()
            time.sleep(2)
    except KeyboardInterrupt:
        print("\nĐang dừng queue và workers, chờ cleanup...", flush=True)
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
        process.wait()
        raise
    finally:
        display_progress()
        print("\nQueue log:", queue_log)
        print("Run root:", RUN_ROOT)

exit_code = process.returncode
status = read_status()
print("Exit code:", exit_code, "| Status:", status.get("status", "not_started"))
if exit_code:
    raise RuntimeError(f"Queue thất bại. Xem traceback/log phía trên và {RUN_ROOT}/status.json")
if status.get("status") != "completed":
    raise RuntimeError("Queue chưa hoàn thành toàn bộ cấu hình.")


## 6. Tổng hợp kết quả
Chỉ tổng hợp sau khi cell train trả quyền điều khiển. Mỗi notebook có một training
seed nên chưa có sample std giữa 5 seeds. Tổng hợp đủ seeds ở bước phân tích sau.
Stochastic là chính, greedy là phụ; báo cáo riêng từng q.


In [ ]:
if not (RUN_ROOT / "manifest.json").is_file():
    raise FileNotFoundError(f"Chưa có manifest: {RUN_ROOT}; queue chưa khởi tạo thành công.")
run_visible(UV + [
    "run", "--no-sync", "python", "-m", "scripts.analysis.matrix_status",
    "--run-root", str(RUN_ROOT),
], cwd=REPO)
print("Báo cáo:", RUN_ROOT / "analysis_summary.json")


## 7. TensorBoard (tùy chọn, sau khi cell train kết thúc)
Dùng TensorBoard trong môi trường dự án. Chạy lại cell sẽ dùng lại server của cell
trước nếu còn sống. Kết quả/log nằm trên Drive, không phụ thuộc thư mục source.


In [ ]:
if "tensorboard_process" not in globals() or tensorboard_process.poll() is not None:
    tensorboard_log = Path("/tmp/acd_tensorboard.log")
    with tensorboard_log.open("ab") as output:
        tensorboard_process = subprocess.Popen(
            UV + ["run", "--no-sync", "tensorboard",
                  "--logdir", str(RUN_ROOT), "--port", "6006", "--host", "0.0.0.0"],
            cwd=REPO, stdout=output, stderr=subprocess.STDOUT,
        )
    time.sleep(3)
if tensorboard_process.poll() is not None:
    print(tensorboard_log.read_text()[-6000:])
    raise RuntimeError("TensorBoard không khởi động được.")
from google.colab import output
output.serve_kernel_port_as_iframe(6006, height=800)
